In [1]:
%load_ext autoreload
%autoreload 2

In [15]:
import sys
import os
sys.path.append('..')
import numpy as np
import pandas as pd
from src.data import process_data_xgboost, load_dataset, get_sample_weights
from src.training import train_xgboost
import mlflow
import joblib
import json

In [3]:
df = load_dataset(True)

In [4]:
X_train, X_test, y_train, y_test = process_data_xgboost(df, 'Transport_mode')

2026/08/15 12:37:45 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/15 12:37:45 INFO mlflow.store.db.utils: Updating database tables


In [7]:
xgbmodels, fold_thresholds = train_xgboost(X_train, y_train, 10, 2000, 0.01, 42, 10, "balanced", get_sample_weights, "xgb_experiment")

Finished_training_1
Finished_training_2
Finished_training_3
Finished_training_4
Finished_training_5
Finished_training_6
Finished_training_7
Finished_training_8
Finished_training_9
Finished_training_10


In [8]:
mlflow.set_tracking_uri(r"sqlite:///C:/Users/USER/Downloads/Projects/mode_choice/notebooks/mlflow.db")
experiment = mlflow.get_experiment_by_name("xgb_experiment")
print(experiment)

<Experiment: artifact_location='file:C:/Users/USER/Downloads/Projects/mode_choice/notebooks/mlruns/1', creation_time=1786811869937, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786811869937, lifecycle_stage='active', name='xgb_experiment', tags={}, trace_location=None, workspace='default'>


In [9]:
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
runs_df.shape  

(16, 78)

In [10]:
row = runs_df.iloc[1]
row.dropna()  

run_id                                           7ed63103535840458fb31c92f5ff9b25
experiment_id                                                                   1
status                                                                   FINISHED
artifact_uri                    file:C:/Users/USER/Downloads/Projects/mode_cho...
start_time                                       2026-08-15 18:25:41.221000+00:00
end_time                                         2026-08-15 18:36:39.478000+00:00
metrics.val_macro_f1                                                     0.607162
metrics.val_accuracy                                                     0.614464
metrics.val_weighted_f1                                                  0.614432
metrics.Fold 9 pr_auc_class1                                             0.522438
metrics.Fold 9 pr_auc_class2                                             0.663675
metrics.Fold 9 pr_auc_class3                                             0.669384
metrics.macro_pr

In [14]:
from pathlib import Path
xgb_dir = Path(os.getcwd()) / "xgboost_models"
xgb_dir.mkdir(parents=True, exist_ok=True)

for i, model in enumerate(xgbmodels):
    joblib.dump(model, xgb_dir / f"xgb_fold_{i+1}.joblib")

In [16]:
thresholds_serializable = {
    k: (v.tolist() if hasattr(v, "tolist") else v)
    for k, v in fold_thresholds.items()
}
with open(xgb_dir / "thresholds.json", "w") as f:
    json.dump(thresholds_serializable, f, indent=4)